# Tutorial 08: Amortized Neural Posterior Estimation with sbi

Simulation-based inference (SBI) with neural networks is a supervised learning approach where a network is trained to learn the mapping between simulated data and the posterior distribution of the underlying model parameters. The framework thus allows us to perform robust statistical inference and derive credibility intervals for parameter estimation with complex simulators like those developed for pulsar population synthesis. Our implementation builds on the [sbi](https://sbi-dev.github.io/sbi/) library ([Tejero-Cantero et al., 2020](https://arxiv.org/abs/2007.09114)).

In this tutorial, we will focus on a specific version of SBI and use the `mlpoppyns/learning/sbi_train.py` script to employ so-called Neural Posterior Estimation (NPE). This allows us to train a neural density estimator on a dataset of samples of simulated neutron star populations to directly approximate the posterior distributions of the input parameters. Note that using the same script, you can also perform [Neural Likelihood Estimation (NLE)](https://proceedings.mlr.press/v89/papamakarios19a/papamakarios19a.pdf) and [Neural Ratio Estimation (NRE)](https://proceedings.mlr.press/v119/hermans20a/hermans20a.pdf). 

To create the dataset of mapped simulations for this training experiment, we can either use Tutorial 04 and 05,  `04_simulation_helper_tutorial.ipynb` and `05_generator_tutorial.ipynb`, respectively, or alternatively we can take advantage of the dataset that is stored in `data/example_generator_magrot`. We also require the output of Tutorial `06_generator_observation_tutorial.ipynb` to produce the density maps associated with the observed population.

In the following, we will perform NPE on our dataset composed of heatmaps or 2D arrays and determine the posterior distributions of the two parameters `P_initial_log10_mean` and `B_initial_log10_mean`. To this end, we will use the script `mlpoppyns/learning/sbi_train.py` as follows:
```commandline
python mlpoppyns/learning/sbi_train.py --configuration config_sbi.json
```
Here, the `config_sbi.json` file contains all the information required to optimize the neural network. If this `CLI` 
argument is left empty, the script will take the default `mlpoppyns/learning/config_sbi.json`.

NOTE: If you would like to infer different parameters than the ones shown in this notebook, you would need to first create a dataset of simulations where these parameters are varied and then train a model to infer them. We refer to the full documentation for more details.

For an application of this inference approach see [Graber et al. (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJ...968...16G/abstract) and [Pardo et al. (2025)](https://arxiv.org/abs/2412.04070).

In [ ]:
import collections
import corner
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys
import torch
import os
import shutil
from matplotlib.ticker import ScalarFormatter
from typing import Tuple
import utilities.plot_settings

## Data folder structure

As a first step, we create the appropriate folder structure required to run the training script. Although this structure is strictly necessary only for the multi-round sequential case, and we are performing a single-round inference in this example, we will follow the multi-round structure for consistency. The folder structure will look like this:

```
output/
└── data/
    ├── test_dataset/
    │   ├── generated_dataset/
    │   │   └── round_0/
    │   └── simulations/
    └── training_dataset/
        ├── generated_dataset/
        │   └── round_0/
        └── simulations/
```
This structure includes separate directories for the training and testing datasets. Within each, there are two subdirectories: `simulations` is for saving the raw simulations, while `generated_dataset` stores the corresponding generated maps. In the sequential case, for each round, folders named `round_<i>` (with i>0) are automatically created inside these subdirectories. Each `round_<i>` folder contains the simulations and generated maps produced for that round.

In the following example, we will use the training and testing data generated in Tutorial 05 which was saved in the directory `tutorials/tutorial_notebooks/output/generator`. If instead, you want to use the dataset located in `data/example_generator_magrot`, change the `training_dataset_path` below to point to that path.

We start by moving the existing training and testing datasets (corresponding to what we refer to throughout as the 0th round) into the newly created folders `output/data/training_dataset/round_0` and `output/data/test_dataset/round_0`, respectively. These will be subsequently used for the single-round training procedure. To keep everything organized under the same directory, we recommend moving the `statistics_train.json` file, which is used to normalize or standardize the parameters, into the data folder as well.

All of these steps will be performed automatically with the following code:

In [ ]:
# Create the folder structure need within the path output.
base_path = "../../tutorials/tutorial_notebooks/output"
!python ../../mlpoppyns/learning/utils/data_folder_struct_sbi.py --base_path {base_path}

In [ ]:
# Move the training and testing datasets, along with statistics_full.json, to their respective directories.
with open(f"config_train_sbi.json", "r") as read_file:
    config_train = json.load(read_file)

training_dataset_path = "../../tutorials/tutorial_notebooks/output/generator"

# Predefined folder names for the round 0 of the training procedure.
train_dst = f"{base_path}/data/training_dataset/generated_dataset"
test_dst = f"{base_path}/data/test_dataset/generated_dataset"
train_dst_round_0 = f"{train_dst}/round_0"
test_dst_round_0 = f"{test_dst}/round_0"

stats_path = f"{base_path}/data/statistics_train.json"

# Copy the training and testing dataset to their respective subfolders.
shutil.copy(
    os.path.join(training_dataset_path, "dataset_train.csv"),
    os.path.join(train_dst_round_0, "dataset_full.csv"),
)
shutil.copy(
    os.path.join(training_dataset_path, "dataset_test.csv"),
    os.path.join(test_dst_round_0, "dataset_full.csv"),
)
shutil.copy(
    os.path.join(training_dataset_path, "statistics_train.json"),
    stats_path,
)

Next, update the `training_data_loader` and `test_data_loader` sections in the configuration file to reflect the new locations of the training and testing datasets. Note that the sbi configuration file, `config_train_sbi.json` in this case, contains specific paths defined for `config_train["training_data_loader"]["dataset_path_round_0"]` and `config_train["test_data_loader"]["dataset_path_round_0"]`, corresponding to the training and testing datasets for round 0. This is done because we assume that the data for round 0 has already been generated before running this script. Since round 0 uses a fixed prior, this data can be reused across multiple experiments, avoiding unnecessary computation and saving resources.

In [ ]:
config_train["training_data_loader"]["dataset_path_round_0"] = train_dst_round_0
config_train["test_data_loader"]["dataset_path_round_0"] = test_dst_round_0

config_train["training_data_loader"]["dataset_path"] = train_dst
config_train["test_data_loader"]["dataset_path"] = test_dst
config_train["test_data_loader"]["statistic_path"] = stats_path

To set up **single-round** inference, simply set `num_rounds` to 1 in the configuration file. Although in this tutorial we show the single-round inference, for **multi-round inference**, you will need to update the following settings in the configuration file:

1. Specify the desired number of rounds using the `num_rounds` parameter.

2. Set the number of simulations to run on the fly for testing and training using the `num_sim` parameter under `test_data_data_loader` and `training_data_loader`, respectively.

3. Indicate whether to use either the `Dask` or `multiprocessing` packages for parallelizing simulation runs by configuring the appropriate option in the general section of the configuration file.

Note that the training and testing datasets for the first round are supposed to be generated before training, i.e., outside the multi-round inference experiment presented here. Therefore, `num_rounds = 2` would mean training the neural network twice, but generating the training and testing datasets only once for the second round.

In [ ]:
config_train["trainer"]["num_rounds"] = 1 

# Save updated config.
with open("config_train_sbi.json", "w") as write_file:
    json.dump(config_train, write_file, indent=4)

# Running the training script

To execute the training experiment, we run the shell command from inside this notebook.

In [ ]:
!python ../../mlpoppyns/learning/sbi_train.py --configuration config_train_sbi.json

## Extracting training results

In the following, we extract the training and validation losses for the two parameters we are predicting to see how they evolve as a function of training epoch during the network optimization process. At this point, we need to specify exactly which trained model we want to load. 

Note that the relevant information is located in the following path in the `tutorial_notebooks` folder, and saved into subdirectories that are named according to the time when the training scripts was launched, i.e., given in the format `YYYYMMDD_HHMMSS`. This path is printed to the standard output (stdout) during execution, as shown above.

In [ ]:
train_output_path = config_train["trainer"]["save_dir"]
print(train_output_path)

To set the paths to the training logs and saved model, we need to change the time stamps in the variables `train_log_output_path` and `model_output_path` below to where our specific training results have been saved. Otherwise, the following examples will not work.

In [ ]:
train_log_output_path = f"{train_output_path}/logs/SBI_ConvolutionMDN/20250821_103802/"  # Change the last time stamp `YYYYMMDD_HHMMSS`.
model_output_path = f"{train_output_path}/models/SBI_ConvolutionMDN/20250821_103802/"  # Change the time stamp `YYYYMMDD_HHMMSS`.

We can now load the corresponding training and validation information. Note that both are contained in the `training_statistics.json` file in the `round_0` directory.

In [ ]:
with open(
    f"{train_log_output_path}round_0/training_statistics.json", "r"
) as read_file:
    learning_data = json.load(read_file)

Plotting the training and validation losses as a function of training epoch for the two parameters we set out to predict.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    learning_data["training_log_probs"]["step"],
    learning_data["training_log_probs"]["value"],
    linestyle="-",
    linewidth=4,
    color="tab:blue",
    rasterized=True,
    label="training",
)
ax.plot(
    learning_data["validation_log_probs"]["step"],
    learning_data["validation_log_probs"]["value"],
    linestyle="-",
    linewidth=4,
    color="tab:orange",
    rasterized=True,
    label="validation",
)

ax.set_xlabel(r"Epoch")
ax.set_ylabel(r"Accuracy")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0)

plt.show()

As shown in the loss plot above, the training and validation losses can diverge noticeably. This behavior is typical when training on a relatively small dataset, as in this tutorial. Based on our experience, achieving stable loss curves generally requires at least 10,000 samples.

## Performing inference with the trained model

Once our SBI pipeline has been trained, it can be used to infer on the observed dataset or on a test dataset and extract posterior distributions of the corresponding pulsar population parameters. The script `mlpoppyns/learning/sbi_infer.py` allows us to take an experiment configuration file, a pre-trained model, and a dataset, and run the inference.

We first update the path to the trained model that we will use to perform the inference.

In [ ]:
config_train["infer"]["load_dir"] = model_output_path

# Save updated configuration.
with open("config_train_sbi.json", "w") as write_file:
    json.dump(config_train, write_file, indent=4)

We then run the inference script as a shell command from inside this notebook.

In [ ]:
# Construct the command.
command = f"python ../../mlpoppyns/learning/sbi_infer.py --configuration config_train_sbi.json"

# Execute the command.
!{command}

NOTE: If this last evaluation finished with a `ValueError` related to the number of samples drawn, the coverage calculation below will not work. We suggest you run the entire notebook again or (if this step still fails) rerun the notebook `05_generator_tutorial` to create a new test dataset. In principle, this problem should not occur if you are training with a realistic number of test samples. Here, we are only using four.

## Extracting inference results

As above, we need to set the path to the specific inference results we want to load. Note that the relevant information is located in the following path in the `tutorial_notebooks` folder, and saved into subdirectories in the format `YYYYMMDD_HHMMSS` as for the training outlined above. As for the training, this path is printed to the standard output (stdout) during execution, as shown above.

In [ ]:
inference_output_path = config_train["infer"]["save_dir"]
observed_dataset_path = "../../tutorials/tutorial_notebooks/output/generator"

# Predefined folder names for the round 0  of the training procedure.
train_dst = f"{base_path}/data/training_dataset/generated_dataset"

stats_path = f"{base_path}/data/statistics_train.json"

# Copy the training and testing dataset to their respective subfolders.
shutil.copy(
    os.path.join(training_dataset_path, "dataset_train.csv"),
    os.path.join(train_dst_round_0, "dataset_full.csv"),
)

To set the paths to the inference results, we need to change the time stamps in the variable `inference_log_output_path` below to where our specific results have been saved. Otherwise, the following examples will not work.

In [ ]:
inference_log_output_path = f"{inference_output_path}/logs/SBI_ConvolutionMDN/20250820_160922/round_0/"  # Change the time stamp `YYYYMMDD_HHMMSS`.

Next, we load the saved posterior samples at the observed data from `config_train["observed_sample"]["dataset_path"]` to produce a corner plot. 

In [ ]:
posterior_samples = torch.load(
    f"{inference_log_output_path}samples_posterior.pt"
)

Then, we rescale the posterior samples to their original physical ranges.

In [ ]:
def import_statistics(stats_path: str) -> Tuple[list,list,list,list]:
    """
    Extracting the mean and standard deviation for all the parameters in the `stats_path` file.

    Args:
        stats_path (str): Path to the file where the statistics are saved.

    Returns:
        (Tuple[list,list,list,list]): Mean, standard deviation, maximun and minimun for the parameters in the `stats_path` file.
    """
    std_list = []
    mean_list = []
    max_list = []
    min_list = []

    with open(stats_path, "r") as json_file:
        data = json.load(json_file)

    for key, value in data.items():
        std_list.append(value["std"])
        mean_list.append(value["mean"])
        max_list.append(value["max"])
        min_list.append(value["min"])

    mean = np.array(mean_list)
    std = np.array(std_list)
    max_list = np.array(max_list)
    min_list = np.array(min_list)

    return mean, std, max_list, min_list

In [ ]:
mean, std, par_max, par_min = import_statistics(
    config_train["training_data_loader"]["statistic_path"]
)

In [ ]:
if config_train["training_data_loader"]["normalize"]:
    posterior_samples = (posterior_samples + par_min) * (par_max - par_min)

elif config_train["training_data_loader"]["standardize"]:
    posterior_samples = posterior_samples * std + mean

For illustration purposes, we also calculate the quantiles of the posterior samples to extract the median values for the parameters and the 95% credibility interval.

In [ ]:
quantile = np.quantile(posterior_samples, [0.025, 0.5, 0.975], axis=0)

param_median = quantile[1]
param_inf = quantile[0]
param_sup = quantile[2]

param_err_inf = param_median - param_inf
param_err_sup = param_sup - param_median

print(
    "predicted B_initial_log10_mean: ",
    f"{param_median[0]} + {param_err_sup[0]} - {param_err_inf[0]}",
)
print(
    "predicted P_initial_log10_mean: ",
    f"{param_median[1]} + {param_err_sup[1]} - {param_err_inf[1]}",
)

In [ ]:
parameter_ranges = [[12, 14], [-1.5, -0.3]]
parameter_labels = [r"$\mu_{\log B}$", r"$\mu_{\log P}$"]

fig = plt.figure(figsize=(10, 10))

figure = corner.corner(
    posterior_samples.numpy(),
    bins=32,
    labels=parameter_labels,
    label_kwargs={"fontsize": 30},
    range=parameter_ranges,
    quantiles=[0.025, 0.5, 0.975],
    levels=(
        1 - np.exp(-0.5),
        1 - np.exp(-2),
        1 - np.exp(-9.0 / 2.0),
    ),  # 1, 2 and 3 sigma levels
    show_titles=True,
    title_kwargs={"fontsize": 30},
    fig=fig,
)
corner.overplot_lines(figure, param_median, color="tab:red")
corner.overplot_points(
    figure,
    param_median[None],
    marker="s",
    color="tab:red",
)

for ax in figure.get_axes():
    ax.tick_params(axis="both", labelsize=22)

## Coverage probability test

To assess the quality of our inference, we also load the coverage probability results. A diagonal coverage would indicate a well-calibrated posterior, while lower lines suggest overconfident and higher lines conservative posteriors. For more details see Appendix B in [Graber et al. (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJ...968...16G/abstract).

NOTE: In this example, we only have 4 test simulations. A larger number of simulation samples would be required to properly perform a coverage test. The following is shown only for illustrative purposes.

In [ ]:
coverage_probability = np.load(
    f"{inference_log_output_path}coverage_probability.npy"
)
betas = np.linspace(0, 1, len(coverage_probability))

Plotting the coverage.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(
    betas,
    coverage_probability,
    color="steelblue",
    linewidth=3,
    label="upper right",
)

ax.plot([0, 1], [0, 1], color="k", linestyle="--")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.set_xlabel(r"Credibility level $1-\alpha$")
ax.set_ylabel(r"Coverage probability")

plt.show()

## Plotting predicted parameters versus ground truths for the test dataset

For each posterior distribution corresponding to a test sample, we plot its 95% confidence interval and median versus the true value to assess the performance of the inference method on the test dataset. The information on all the posterior samples for each instance in the test dataset are stored in `posterior_samples_test_data.npz`.

In [ ]:
data = np.load(f"{inference_log_output_path}posterior_samples_test_data.npz")
true_values = data["true_values"]
test_posterior_samples = data["posterior_samples"]

We again rescale the samples according to the method used.

In [ ]:
if config_train["training_data_loader"]["normalize"]:
    test_posterior_samples = (test_posterior_samples + par_min) * (
        par_max - par_min
    )
    true_values = (true_values + par_min) * (par_max - par_min)

elif config_train["training_data_loader"]["standardize"]:
    test_posterior_samples = test_posterior_samples * std + mean
    true_values = true_values * std + mean


In [ ]:
num_samples = test_posterior_samples.shape[0]
num_params = test_posterior_samples.shape[2]

print(num_samples, num_params)

# Defining the confidence interval.
ci_prob = 0.95

In [ ]:
# Loop through each of the test samples.
for param_idx in range(num_params):
    
    # Extract the true values for each test samples.
    true_vals = true_values[:, param_idx]
    medians = []
    lowers = []
    uppers = []

    for i in range(num_samples):
        
        # Compute the median and confidence interval for the approximated posterior distribution of each test sample.
        posterior = test_posterior_samples[i, :, param_idx]
        median = np.median(posterior)

        lower_percentile = (1 - ci_prob) / 2 * 100
        upper_percentile = (1 + ci_prob) / 2 * 100

        lower = np.percentile(posterior, lower_percentile)
        upper = np.percentile(posterior, upper_percentile)

        medians.append(median)
        lowers.append(lower)
        uppers.append(upper)

    medians = np.array(medians)
    lowers = np.array(lowers)
    uppers = np.array(uppers)

    yerr = np.vstack([medians - lowers, uppers - medians])

    plt.figure(figsize=(10, 5))

    plt.errorbar(
        true_vals,
        medians,
        yerr=yerr,
        fmt="none",
        ecolor="lightblue",
        alpha=0.8,
        capsize=3,
        label=f"{ci_prob*100}% CI",
    )

    plt.scatter(
        true_vals,
        medians,
        color="green",
        marker="x",
        label="Posterior median",
        s=20,
    )

    plt.plot(
        [true_vals.min(), true_vals.max()],
        [true_vals.min(), true_vals.max()],
        "k--",
    )

    plt.xlabel("True value", fontsize=15)
    plt.ylabel("Predicted value", fontsize=15)
    plt.title(parameter_labels[param_idx], fontsize=15)
    
    # Make tick labels smaller.
    plt.tick_params(axis='both', which='major', labelsize=10)
    
    plt.grid(True)
    plt.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=15)
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.show()


In the plot above, we show the true values versus the inferred parameters for each of the test samples. The median is shown with a green cross, and the 95% confidence interval is represented by a blue solid line. The dashed black line indicates the identity line, meaning that a well-calibrated posterior should follow this line, i.e., the inferred parameters should be as close as possible to the ground truth. In this example, due to the limited number of training samples, the confidence intervals are relatively large, and for some test samples, the predicted values deviate significantly from the true values.